# Credit Card Fraud Dataset — Data Cleaning & Preparation
**Dataset:** `creditcard.csv` (284,807 transactions, 31 columns)
**Goal:** Take the raw dataset through a full audit → clean → summary pipeline using pandas, with a documented decision for every step.


## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset exactly as provided
raw_df = pd.read_csv('creditcard.csv')

# Keep an untouched copy so we can compare "before" vs "after" later
df = raw_df.copy()

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()


Loaded 284,807 rows and 31 columns


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Initial Data Audit
Before changing anything, we need to know: how big is the dataset, what type is each
column stored as, and where (if anywhere) are values missing.


In [2]:
# --- Shape ---
print("Shape (rows, columns):", df.shape)

# --- Data types ---
print("\nData types:")
print(df.dtypes)

# --- Missing values per column ---
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df) * 100).round(3)
})
print("\nMissing values per column:")
print(missing_summary)


Shape (rows, columns): (284807, 31)

Data types:
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Missing values per column:
        missing_count  missing_pct
Time                0          0.0
V1                  0          0.0
V2                  0          0.0
V3                  0          0.0
V4                  0          0.0
V5                  0          0.0
V6                  0          0.0
V7                  0          0.0
V8                  0          0.0
V

In [3]:
# --- Snapshot of "before" state, used later in the before/after summary ---
before_summary = {
    'rows': df.shape[0],
    'columns': df.shape[1],
    'total_missing': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'dtypes': df.dtypes.value_counts().to_dict()
}
before_summary


{'rows': 284807,
 'columns': 31,
 'total_missing': 0,
 'duplicate_rows': 1081,
 'dtypes': {dtype('float64'): 30, dtype('int64'): 1}}

## 3. Clean

### 3.1 Handle Missing Values
**Audit result:** the missing-value summary above shows **zero missing values in every
column** (this is a curated, pre-processed research dataset, so there is nothing to
drop/impute/flag).

**Decision:** no imputation or row-dropping is needed for missingness. We still write
the handling logic below so the pipeline is robust if this script is ever re-run on a
raw export that *does* contain gaps (e.g., a future batch of transactions with a
missing `Amount`).


In [4]:
# Defensive missing-value handling (no-op on this dataset, but documents the decision):
# - Amount (numeric, right-skewed): if missing, impute with the MEDIAN, not the mean,
#   because Amount has extreme outliers that would pull the mean up.
# - V1-V28 (PCA components): if missing, impute with 0 (the PCA components are
#   approximately mean-zero by construction, so 0 is the "no information" value).
# - Class (target label): if missing, DROP the row — we cannot train/validate a
#   fraud model on a transaction with an unknown label.

if df['Amount'].isna().any():
    df['Amount'] = df['Amount'].fillna(df['Amount'].median())

pca_cols = [c for c in df.columns if c.startswith('V')]
df[pca_cols] = df[pca_cols].fillna(0)

df = df.dropna(subset=['Class'])

print("Missing values remaining:", df.isna().sum().sum())


Missing values remaining: 0


### 3.2 Remove Duplicate Records
An exact duplicate row (identical values across all 31 columns, including all 28 PCA
components) is extremely unlikely to occur by chance for two genuinely different
transactions — it almost always indicates the same transaction was captured/exported
twice.

**Decision:** drop exact duplicate rows, keeping the first occurrence.


In [5]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows found: {dup_count}")

df = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Rows after removing duplicates: {len(df):,}")


Exact duplicate rows found: 1081


Rows after removing duplicates: 283,726


### 3.3 Fix Data Types
- **`Time`** is stored as `int64` (seconds elapsed since the first transaction in the
  dataset). Seconds-since-start isn't a real timestamp, but it IS a clock — so we
  convert it into a more usable **`Hour_of_Day`** feature (0–23), since the dataset
  spans about 48 hours and fraud analysts typically care about time-of-day patterns,
  not raw elapsed seconds.
- **`Class`** is stored as `int64` (0/1) but is actually a **categorical label**, not a
  quantity you'd ever average or sum meaningfully. We convert it to pandas `category`
  dtype, which is both correct typing and more memory-efficient.
- **`Amount`** is already a proper float, so no change needed there.


In [6]:
# Convert elapsed seconds into hour-of-day (0-23), wrapping every 24h
df['Hour_of_Day'] = ((df['Time'] // 3600) % 24).astype(int)

# Class is categorical (label), not a continuous integer -> convert dtype
df['Class'] = df['Class'].astype('category')

print(df[['Time', 'Hour_of_Day', 'Class']].dtypes)
df[['Time', 'Hour_of_Day', 'Class']].head()


Time            float64
Hour_of_Day       int64
Class          category
dtype: object


,Time,Hour_of_Day,Class
0,0.0,0,0
1,0.0,0,0
2,1.0,0,0
3,1.0,0,0
4,2.0,0,0


### 3.4 Standardize Categorical Values
The only categorical field in this dataset is `Class`, and because it's a numeric-coded
label (0/1) there's no free-text inconsistency to fix here (no "NY" vs "New York"
style problem exists in this particular dataset).

**Decision:** to make the category genuinely standardized and self-documenting for
anyone reading the cleaned file, we add a parallel human-readable label column
(`Class_Label`) mapping `0 -> 'Legitimate'` and `1 -> 'Fraud'`, and lock it to a fixed
category order so it always sorts/displays consistently.


In [7]:
class_map = {0: 'Legitimate', 1: 'Fraud'}
df['Class_Label'] = df['Class'].astype(int).map(class_map)
df['Class_Label'] = pd.Categorical(df['Class_Label'], categories=['Legitimate', 'Fraud'])

print(df['Class_Label'].value_counts())


Class_Label
Legitimate    283253
Fraud            473
Name: count, dtype: int64


### 3.5 Handle Outliers
`Amount` is heavily right-skewed: 75% of transactions are under \$77, but the maximum
is \$25,691.

**Decision:** we do **NOT** remove or cap these extreme values in the main dataset.
In fraud detection, unusually large (or unusually small) amounts are often exactly the
signal the business cares about — deleting them risks throwing away real fraud cases.
Instead we:
1. Flag extreme values in a new `Amount_Outlier` column (outside 1.5×IQR) so analysts
   can filter on it without losing the underlying data.
2. Provide a separate `Amount_Capped` column (winsorized at the 99.9th percentile) for
   any downstream model that is sensitive to extreme values (e.g., linear regression),
   while leaving the original `Amount` untouched for models that aren't (e.g., tree-based
   models, or the fraud investigation team who need the true dollar figure).


In [8]:
q1, q3 = df['Amount'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df['Amount_Outlier'] = (df['Amount'] < lower_bound) | (df['Amount'] > upper_bound)
print(f"Flagged outliers: {df['Amount_Outlier'].sum():,} rows "
      f"({df['Amount_Outlier'].mean()*100:.2f}% of data)")

cap_value = df['Amount'].quantile(0.999)
df['Amount_Capped'] = df['Amount'].clip(upper=cap_value)

print(f"99.9th percentile cap value: {cap_value:.2f}")
df[['Amount', 'Amount_Capped', 'Amount_Outlier']].describe()


Flagged outliers: 31,685 rows (11.17% of data)
99.9th percentile cap value: 3004.75


,Amount,Amount_Capped
count,283726.000000,283726.000000
mean,88.472687,86.974443
std,250.399437,215.016839
min,0.000000,0.000000
25%,5.600000,5.600000
50%,22.000000,22.000000
75%,77.510000,77.510000
max,25691.160000,3004.752000


## 4. Before / After Summary

In [9]:
after_summary = {
    'rows': df.shape[0],
    'columns': df.shape[1],
    'total_missing': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'dtypes': df.dtypes.value_counts().to_dict()
}

comparison = pd.DataFrame({
    'Before': [before_summary['rows'], before_summary['columns'],
               before_summary['total_missing'], before_summary['duplicate_rows']],
    'After':  [after_summary['rows'], after_summary['columns'],
               after_summary['total_missing'], after_summary['duplicate_rows']]
}, index=['Row count', 'Column count', 'Total missing values', 'Duplicate rows'])

comparison


,Before,After
Row count,284807,283726
Column count,31,35
Total missing values,0,0
Duplicate rows,1081,0


In [10]:
print("Before dtypes:", before_summary['dtypes'])
print("After dtypes: ", after_summary['dtypes'])

print("\nNew columns added during cleaning:")
new_cols = set(df.columns) - set(raw_df.columns)
print(sorted(new_cols))


Before dtypes: {dtype('float64'): 30, dtype('int64'): 1}
After dtypes:  {dtype('float64'): 31, CategoricalDtype(categories=[0, 1], ordered=False, categories_dtype=int64): 1, dtype('int64'): 1, CategoricalDtype(categories=['Legitimate', 'Fraud'], ordered=False, categories_dtype=str): 1, dtype('bool'): 1}

New columns added during cleaning:
['Amount_Capped', 'Amount_Outlier', 'Class_Label', 'Hour_of_Day']


## 5. Export Cleaned Dataset

In [11]:
df.to_csv('creditcard_cleaned.csv', index=False)
print(f"Saved cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]} columns -> creditcard_cleaned.csv")


Saved cleaned dataset: 283,726 rows x 35 columns -> creditcard_cleaned.csv
